# La stessa rete con TensorFlow e Keras

Materiale originale **AI Lab**, non una lezione Kaggle. Codice in inglese; attività e spiegazioni in italiano.

Esegui le celle in ordine con **Maiusc + Invio**. Salva una copia prima di modificare il codice.


## Requisiti
Python 3.11/3.12 con `ai_lab/requirements/deep-learning.txt`. CPU sufficiente, nessun dataset remoto. **Esecuzione TensorFlow locale non verificata in questa sessione**: il pacchetto non è disponibile e la rete impedisce di installarlo. Il workflow scientifico è predisposto per verificarla in un ambiente compatibile.


In [ ]:
import os
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
features = rng.uniform(-1, 1, (800, 2)).astype('float32')
targets = ((features[:, 0] * features[:, 1]) > 0).astype('float32').reshape(-1, 1)
order = rng.permutation(len(features))
features, targets = features[order], targets[order]
X_train, X_valid, X_test = features[:480], features[480:640], features[640:]
y_train, y_valid, y_test = targets[:480], targets[480:640], targets[640:]
assert not np.isnan(features).any()
print('Training shape:', X_train.shape, 'Target shape:', y_train.shape)


In [ ]:
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16, activation="tanh"),
    layers.Dense(8, activation="tanh"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
              loss="binary_crossentropy", metrics=["binary_accuracy"])
options = tf.data.Options()
options.threading.private_threadpool_size = 1
train_data = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(480, seed=42).batch(32).with_options(options)
valid_data = tf.data.Dataset.from_tensor_slices((X_valid, y_valid)).batch(32).with_options(options)
history = model.fit(train_data, validation_data=valid_data, epochs=80,
                    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)
predictions = model(X_test, training=False).numpy()
accuracy = float(np.mean((predictions >= 0.5) == y_test))
print("Test accuracy:", accuracy)
assert accuracy >= 0.85
model.save("xor-model.keras")


## Esercizi
Confronta il codice con il laboratorio NumPy. Aggiungi Dropout da 0.1 fra gli strati nascosti, eseguendo nuovamente dall’inizio. Confronta le curve su validazione prima del test.

## Riepilogo
Keras automatizza gradienti e aggiornamenti. Dati, loss e protocollo di valutazione restano responsabilità di chi progetta il modello.
